In [2]:
import sys
import pathlib

# set pythonpath to the main module directory
module_dir = pathlib.Path("..").parent.resolve().parent
if str(module_dir) not in sys.path:
    sys.path.append(str(module_dir))

In [3]:
import pandas as pd

clean_results_path = "../clean_results/res_clean.json"
dirty_results_path = "../res.json"

clean_results = pd.read_json(clean_results_path, orient="records")
dirty_results = pd.read_json(dirty_results_path, orient="records")

display(clean_results)
display(dirty_results)

In [4]:
def parse_path_clean(path: str) -> dict:
    parts = path.split("/")
    model_name = parts[-2]
    train_dataset = None
    layer_name = None
    exp_name = None

    return {"model_name": model_name, "train_dataset": train_dataset, "layer_name": layer_name, "exp_name": exp_name}


def parse_path_dirty(path: str) -> dict:
    parts = path.split("/")
    model_name = parts[-5]
    train_dataset = parts[-4]
    layer_name = parts[-3]
    exp_name = parts[-2]

    return {"model_name": model_name, "train_dataset": train_dataset, "layer_name": layer_name, "exp_name": exp_name}


# # now use parse_path to add corresponding columns to the dataframe
# parsed = results["path"].apply(parse_path)
# results = pd.concat([results, parsed.apply(pd.Series)], axis=1)

# # results = results.drop(columns=["path", "file"])

# results.sample(20)

parsed = clean_results.apply(lambda row: pd.Series(parse_path_clean(row["path"])), axis=1)
clean_results = pd.concat([clean_results, parsed], axis=1)


parsed = dirty_results.apply(lambda row: pd.Series(parse_path_dirty(row["path"])), axis=1)
dirty_results = pd.concat([dirty_results, parsed], axis=1)

display(clean_results)
display(dirty_results)

In [5]:
def parse_experiment(exp_name: str) -> dict:
    kl_value = None
    lr_value = None
    early_stop = True

    if "no-early-stop" in exp_name:
        early_stop = False

    if "baseline" in exp_name or "no-early-stop" in exp_name:
        lr_value = 0.1
        kl_value = 1.0

    if "small-lr" in exp_name:
        lr_value = 0.02
        kl_value = 1.0

    if "medium-lr" in exp_name:
        lr_value = 0.04
        kl_value = 1.0

    if "large-lr" in exp_name:
        lr_value = 0.25
        kl_value = 1.0

    if "small-kl" in exp_name:
        lr_value = 0.1
        kl_value = 0.5

    if "high-kl" in exp_name:
        lr_value = 0.1
        kl_value = 2.0

    # otherwise, the kl should appear as .._kl=<number>-..

    if "kl=" in exp_name:
        lr_value = 0.1

        kl_part = [part for part in exp_name.split("_") if part.startswith("kl=")]
        if not kl_part:
            raise ValueError(f"Could not find kl value in experiment name: {exp_name}")

        kl_value_str = kl_part[0].split("=")[1].split("-")[0]
        kl_value = float(kl_value_str)

    if lr_value is None or kl_value is None:
        raise ValueError(f"Could not parse experiment name: {exp_name}")

    return {"kl_value": kl_value, "lr_value": lr_value, "early_stop": early_stop}


parsed_exp = dirty_results["exp_name"].apply(lambda x: pd.Series(parse_experiment(x)))
dirty_results = pd.concat([dirty_results, parsed_exp], axis=1)

display(dirty_results)


In [6]:
def intersect_on_columns(df1: pd.DataFrame, df2: pd.DataFrame, on: list[str]) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Filter two dataframes to only keep rows whose values in `on` columns
    appear in both dataframes. Rows whose key is found only in one dataframe
    are discarded.

    Parameters
    ----------
    df1, df2 : pd.DataFrame
        Input dataframes to filter.
    on : list[str]
        Column name(s) to use as the intersection key.

    Returns
    -------
    (df1_filtered, df2_filtered) : tuple of pd.DataFrame
    """
    common_keys = pd.merge(
        df1[on].drop_duplicates(),
        df2[on].drop_duplicates(),
        on=on,
    )
    df1_filtered = df1.merge(common_keys, on=on)
    df2_filtered = df2.merge(common_keys, on=on)
    return df1_filtered, df2_filtered


In [ ]:
dirty_results["model_name"].unique()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

dirty_results_filtered = dirty_results[
    (dirty_results["kl_value"] == 0.5)
    & (dirty_results["early_stop"] == True)
    & (dirty_results["lr_value"] == 0.1)
    & (dirty_results["model_name"] == "Llama-2-7b-chat-hf")
].copy()

dirty_results_filtered["source"] = "finetuned"
clean_results_copy = clean_results.copy()
clean_results_copy["source"] = "baseline"

groups = dirty_results_filtered[["model_name", "train_dataset", "layer_name"]].drop_duplicates()

for _, group in groups.iterrows():
    model_name, train_dataset, layer_name = group["model_name"], group["train_dataset"], group["layer_name"]

    dirty_group = dirty_results_filtered[
        (dirty_results_filtered["model_name"] == model_name)
        & (dirty_results_filtered["train_dataset"] == train_dataset)
        & (dirty_results_filtered["layer_name"] == layer_name)
    ]
    clean_group = clean_results_copy[clean_results_copy["model_name"] == model_name]

    dirty_group, clean_group = intersect_on_columns(dirty_group, clean_group, on=["benchmark", "metric"])

    # display(dirty_group)
    # display(clean_group)

    ratio_df = dirty_group.merge(clean_group, on=["benchmark", "metric"], suffixes=("_dirty", "_clean"))
    ratio_df["value"] = (ratio_df["value_dirty"] - ratio_df["value_clean"]) / ratio_df["value_clean"]
    ratio_df["benchmark_metric"] = ratio_df["benchmark"] + " / " + ratio_df["metric"]

    ratio_df = ratio_df.sort_values("value", ascending=True)
    order = ratio_df["benchmark_metric"].tolist()

    n_unique = ratio_df["benchmark_metric"].nunique()

    #print nomber of nan values in the value column
    n_nan = ratio_df["value"].isna().sum()
    print(f"Number of NaN values in 'value' column: {n_nan} out of {n_unique} unique benchmark/metric combinations")

    g = sns.catplot(
        data=ratio_df,
        x="benchmark_metric",
        y="value",
        kind="bar",
        # order=order,
        # height=5,
        aspect=3,
        
    )

    ax = g.ax
    ax.set_title(f"Model: {model_name} | Dataset: {train_dataset} | Layer: {layer_name}", pad=6)
    ax.set_xlabel("")
    ax.set_ylabel("Value")
    ax.tick_params(axis="x", rotation=90, labelsize=6.5)
    ax.tick_params(axis="y", labelsize=8)
    ax.grid(axis="y", alpha=0.3)
    ax.legend(title="Source", fontsize=8, title_fontsize=8)

    plt.tight_layout(pad=0.5)
    plt.show()

    # break
